In [ ]:
# note everything below and in every cell below is the same as in analysis.ipynb
# but just took the part that is relevant to the imbalance handling to be here

# to see the class imbalance in the dataset before train/test split
# findings from this cell:
# that it expect roughly 31% low and 69% high
# so a model that always predicts high can look accurate ~69% while failing on low salaries

import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(".").resolve()))
from utils import load_ml_dataframe, make_high_salary_label

ml_df = load_ml_dataframe()
y = make_high_salary_label(ml_df)

imbalance_table = pd.DataFrame(
    {
        "class": ["low (0)", "high (1)"],
        "count": [int((y == 0).sum()), int((y == 1).sum())],
        "share": [
            round((y == 0).mean(), 4),
            round((y == 1).mean(), 4),
        ],
    }
)
imbalance_table

,class,count,share
0,low (0),9104,0.3132
1,high (1),19961,0.6868


In [ ]:
# splits the data into training and testing parts
# 80% train / 20% test with stratify=y
# stratify keeps about 31% low and 69%
# to avoid containing too many high salary rows or too many low salary rows

# and then prints a table showing the number of low and high salary rows in the test set (support)

from utils import get_classification_split

# step 13
X_train, X_test, y_train, y_test = get_classification_split()

support_table = pd.DataFrame(
    {
        "class": ["low (0)", "high (1)"],
        "support_in_test_set": [
            int((y_test == 0).sum()),
            int((y_test == 1).sum()),
        ],
    }
)
support_table

,class,support_in_test_set
0,low (0),1821
1,high (1),3992


In [ ]:
# this trains the same logstic model also as in analysis.ipynb on X_train, y_train from the cell above
# 0 = low salary & 1 = high salary
# after trainging it predicts labels for X_test
# then it compares the predictions with the real labels

# then this prints precision, recall, f1 score, support

# and it saves the results to data/results_logistic_regression.csv

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from utils import ROOT, build_pipeline, evaluate_classification

log_pipeline = build_pipeline(
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)  # step 13
)
log_pipeline.fit(X_train, y_train)
log_pred = log_pipeline.predict(X_test)

log_metrics, _ = evaluate_classification(y_test, log_pred, model_name="logistic_regression")
pd.DataFrame([log_metrics]).to_csv(ROOT / "data" / "results_logistic_regression.csv", index=False)
#print(classification_report(y_test, log_pred, target_names=["low", "high"]))

              precision    recall  f1-score   support

         low       0.41      0.61      0.49      1821
        high       0.77      0.59      0.67      3992

    accuracy                           0.60      5813
   macro avg       0.59      0.60      0.58      5813
weighted avg       0.65      0.60      0.61      5813



In [ ]:
# same as cell above but using random forest :)

# this trains the same random forest model as in analysis.ipynb on X_train, y_train from the cell above
# 0 = low salary & 1 = high salary
# random forest uses many decision trees, and each tree votes for the final class
# class_weight="balanced" helps the model care more about the minority class, which is low salary

# after training it predicts labels for X_test
# then it compares the predictions with the real labels

# then this prints precision, recall, f1 score, support

# and it saves the results to data/results_random_forest_classifier.csv

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from utils import ROOT, build_pipeline, evaluate_classification

rf_pipeline = build_pipeline(
    RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=10,
        class_weight="balanced",
    )
)
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

rf_metrics, _ = evaluate_classification(y_test, rf_pred, model_name="random_forest_classifier")
pd.DataFrame([rf_metrics]).to_csv(ROOT / "data" / "results_random_forest_classifier.csv", index=False)
print(classification_report(y_test, rf_pred, target_names=["low", "high"]))

              precision    recall  f1-score   support

         low       0.41      0.59      0.49      1821
        high       0.77      0.62      0.69      3992

    accuracy                           0.61      5813
   macro avg       0.59      0.60      0.59      5813
weighted avg       0.66      0.61      0.62      5813

